In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully!")

In [ ]:
import os

# check what's inside input folder
print(os.listdir("/kaggle/input/"))

In [ ]:
import os

for folder in os.listdir("/kaggle/input/"):
    print(folder)
    print(os.listdir(f"/kaggle/input/{folder}"))

In [ ]:
import os

base = "/kaggle/input/datasets/aditya8283"
print(os.listdir(base))

In [ ]:
for folder in os.listdir(base):
    print(folder)
    print(os.listdir(f"{base}/{folder}"))

In [ ]:
import pandas as pd

demand = pd.read_excel("/kaggle/input/datasets/aditya8283/predictive-paradox/PGCB_date_power_demand.xlsx")
weather = pd.read_excel("/kaggle/input/datasets/aditya8283/predictive-paradox/weather_data.xlsx")
economic = pd.read_csv("/kaggle/input/datasets/aditya8283/predictive-paradox/economic_full_1.csv")

print("demand rows:", len(demand))
print("weather rows:", len(weather))
print("economic rows:", len(economic))

In [ ]:
demand.head()

In [ ]:
weather.head()

In [ ]:
economic.head()

In [ ]:
print("Demand columns:", demand.columns.tolist())
print("Weather columns:", weather.columns.tolist())
print("Economic columns:", economic.columns.tolist())

In [ ]:
print(demand.shape)
print(demand.dtypes)
print(demand.isnull().sum())

In [ ]:
demand.head(10)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15,4))
plt.plot(demand['demand_mw'], color='blue', linewidth=0.5)
plt.title("Raw Electricity Demand")
plt.ylabel("MW")
plt.show()

In [ ]:
demand['demand_mw'].describe()

In [ ]:
print(weather.shape)
print(weather.isnull().sum())
weather.head()

In [ ]:

demand['datetime'] = pd.to_datetime(demand['datetime'])


demand = demand.sort_values('datetime').reset_index(drop=True)


demand = demand.drop_duplicates(subset='datetime')

print("After cleaning:", demand.shape)
print(demand['datetime'].min(), "to", demand['datetime'].max())

In [ ]:
# calculate rolling average and std
rolling_mean = demand['demand_mw'].rolling(window=24).mean()
rolling_std = demand['demand_mw'].rolling(window=24).std()


upper = rolling_mean + 3 * rolling_std
lower = rolling_mean - 3 * rolling_std


demand.loc[(demand['demand_mw'] > upper) | (demand['demand_mw'] < lower), 'demand_mw'] = np.nan


demand['demand_mw'] = demand['demand_mw'].fillna(method='ffill')

print("Spikes removed!")
print(demand['demand_mw'].describe())

In [ ]:
plt.figure(figsize=(15,4))
plt.plot(demand['demand_mw'], color='green', linewidth=0.5)
plt.title("Cleaned Electricity Demand")
plt.ylabel("MW")
plt.show()

In [ ]:

weather = pd.read_excel("/kaggle/input/datasets/aditya8283/predictive-paradox/weather_data.xlsx", skiprows=3)


weather.head()

In [ ]:
print(weather.columns.tolist())
print(weather.shape)

In [ ]:

weather = weather.rename(columns={'time': 'datetime'})

weather['datetime'] = pd.to_datetime(weather['datetime'])

weather = weather.sort_values('datetime').reset_index(drop=True)

print("Weather cleaned!")
print(weather['datetime'].min(), "to", weather['datetime'].max())
weather.head()

In [ ]:

df = pd.merge(demand[['datetime', 'demand_mw']], weather, on='datetime', how='inner')

print("Merged shape:", df.shape)
df.head()

In [ ]:

economic = pd.read_csv("/kaggle/input/datasets/aditya8283/predictive-paradox/economic_full_1.csv")


eco_bd = economic[economic['Country Name'] == 'Bangladesh']

print(eco_bd.shape)
eco_bd.head()

In [ ]:

eco_melted = eco_bd.melt(
    id_vars=['Country Name', 'Indicator Name', 'Indicator Code'],
    var_name='year',
    value_name='value'
)

print(eco_melted.head(10))
print(eco_melted['Indicator Name'].unique())

In [1]:
# see all unique country names
print(economic['Country Name'].unique())

NameError: name 'economic' is not defined

In [ ]:

mask = economic['Country Name'].str.contains('Bang', case=False, na=False)
print(economic[mask]['Country Name'].unique())

In [ ]:
economic.head(20)

In [ ]:
print(economic.shape)
print(economic.columns.tolist())

In [ ]:
print(economic['Indicator Name'].unique()[:20])

In [ ]:
# pick only useful indicators for electricity demand
useful = [
    'GDP (current US$)',
    'Population, total',
    'Urban population',
    'Industry (including construction), value added (% of GDP)',
    'Electric power consumption (kWh per capita)'
]

eco_filtered = economic[economic['Indicator Name'].isin(useful)]
print(eco_filtered.shape)
print(eco_filtered['Indicator Name'].unique())

In [ ]:

eco_filtered2 = economic[economic['Indicator Name'].isin(useful)].copy()


year_cols = [str(y) for y in range(2010, 2026)]
eco_small = eco_filtered2[['Indicator Name'] + year_cols].copy()

# set indicator as index then transpose
eco_small = eco_small.set_index('Indicator Name').T.reset_index()
eco_small = eco_small.rename(columns={'index': 'year'})
eco_small['year'] = eco_small['year'].astype(int)

print(eco_small.shape)
eco_small.head()

In [ ]:
df['year'] = df['datetime'].dt.year

df = pd.merge(df, eco_small, on='year', how='left')

print("Final shape:", df.shape)
df.head()

In [ ]:
# extract time features from datetime
df['hour'] = df['datetime'].dt.hour
df['day_of_week'] = df['datetime'].dt.dayofweek
df['month'] = df['datetime'].dt.month
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

print("Time features added!")

In [ ]:

df = df.sort_values('datetime').reset_index(drop=True)


df['lag_1h'] = df['demand_mw'].shift(1)
df['lag_2h'] = df['demand_mw'].shift(2)
df['lag_24h'] = df['demand_mw'].shift(24)
df['lag_48h'] = df['demand_mw'].shift(48)
df['lag_168h'] = df['demand_mw'].shift(168)

print("Lag features added!")

In [ ]:
df['rolling_mean_3h'] = df['demand_mw'].shift(1).rolling(3).mean()
df['rolling_mean_24h'] = df['demand_mw'].shift(1).rolling(24).mean()
df['rolling_std_24h'] = df['demand_mw'].shift(1).rolling(24).std()

print("Rolling features added!")

In [ ]:

df['target'] = df['demand_mw'].shift(-1)


df = df.dropna().reset_index(drop=True)

print("Final shape:", df.shape)
df.head()

In [ ]:

train = df[df['datetime'] < '2023-01-01']
test = df[df['datetime'] >= '2023-01-01']

print("Train size:", train.shape)
print("Test size:", test.shape)

In [ ]:

drop_cols = ['datetime', 'demand_mw', 'target', 'year']

X_train = train.drop(columns=drop_cols)
y_train = train['target']

X_test = test.drop(columns=drop_cols)
y_test = test['target']

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

# LOAD DATA
demand = pd.read_excel("/kaggle/input/datasets/aditya8283/predictive-paradox/PGCB_date_power_demand.xlsx")
weather = pd.read_excel("/kaggle/input/datasets/aditya8283/predictive-paradox/weather_data.xlsx", skiprows=3)
economic = pd.read_csv("/kaggle/input/datasets/aditya8283/predictive-paradox/economic_full_1.csv")

# CLEAN DEMAND
demand['datetime'] = pd.to_datetime(demand['datetime'])
demand = demand.sort_values('datetime').drop_duplicates(subset='datetime').reset_index(drop=True)
rolling_mean = demand['demand_mw'].rolling(24).mean()
rolling_std = demand['demand_mw'].rolling(24).std()
demand.loc[(demand['demand_mw'] > rolling_mean + 3*rolling_std), 'demand_mw'] = np.nan
demand['demand_mw'] = demand['demand_mw'].fillna(method='ffill')

# CLEAN WEATHER
weather = weather.rename(columns={'time': 'datetime'})
weather['datetime'] = pd.to_datetime(weather['datetime'])
weather = weather.sort_values('datetime').reset_index(drop=True)

# MERGE
df = pd.merge(demand[['datetime', 'demand_mw']], weather, on='datetime', how='inner')

# ECONOMIC
useful = ['GDP (current US$)', 'Population, total', 'Urban population']
eco_filtered = economic[economic['Indicator Name'].isin(useful)].copy()
year_cols = [str(y) for y in range(2010, 2026)]
eco_small = eco_filtered[['Indicator Name'] + year_cols].set_index('Indicator Name').T.reset_index()
eco_small = eco_small.rename(columns={'index': 'year'})
eco_small['year'] = eco_small['year'].astype(int)
df['year'] = df['datetime'].dt.year
df = pd.merge(df, eco_small, on='year', how='left')

# CLEAN COLUMN NAMES
df.columns = df.columns.str.replace(r'[^\w]', '_', regex=True).str.strip('_')

# FEATURES
df = df.sort_values('datetime').reset_index(drop=True)
df['hour'] = df['datetime'].dt.hour
df['day_of_week'] = df['datetime'].dt.dayofweek
df['month'] = df['datetime'].dt.month
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['lag_1h'] = df['demand_mw'].shift(1)
df['lag_2h'] = df['demand_mw'].shift(2)
df['lag_24h'] = df['demand_mw'].shift(24)
df['lag_168h'] = df['demand_mw'].shift(168)
df['rolling_mean_3h'] = df['demand_mw'].shift(1).rolling(3).mean()
df['rolling_mean_24h'] = df['demand_mw'].shift(1).rolling(24).mean()
df['rolling_std_24h'] = df['demand_mw'].shift(1).rolling(24).std()
df['target'] = df['demand_mw'].shift(-1)
df = df.dropna().reset_index(drop=True)

# CHECK DATES
print("Date range:", df['datetime'].min(), "to", df['datetime'].max())
print("Total rows:", len(df))

# SPLIT - use last 20% as test
split_index = int(len(df) * 0.8)
train = df.iloc[:split_index]
test = df.iloc[split_index:]

drop_cols = ['datetime', 'demand_mw', 'target', 'year']
X_train = train.drop(columns=drop_cols, errors='ignore')
y_train = train['target']
X_test = test.drop(columns=drop_cols, errors='ignore')
y_test = test['target']

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

# LIGHTGBM
lgb_model = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, random_state=42)
lgb_model.fit(X_train, y_train)
lgb_pred = lgb_model.predict(X_test)
lgb_mape = (abs(y_test - lgb_pred) / y_test).mean() * 100

# RANDOM FOREST
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_mape = (abs(y_test - rf_pred) / y_test).mean() * 100

# LINEAR REGRESSION
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)
lr_mape = (abs(y_test - lr_pred) / y_test).mean() * 100

# RESULTS
print("===== MODEL COMPARISON =====")
print("LightGBM MAPE      :", round(lgb_mape, 2), "%")
print("Random Forest MAPE :", round(rf_mape, 2), "%")
print("Linear Regression  :", round(lr_mape, 2), "%")
print("============================")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15,4))
plt.plot(y_test.values[:500], label='Actual', color='blue')
plt.plot(lgb_pred[:500], label='LightGBM', color='red', alpha=0.7)
plt.title("Actual vs Predicted Demand")
plt.legend()
plt.show()

In [ ]:
import pandas as pd

feat_imp = pd.Series(lgb_model.feature_importances_, index=X_train.columns)
feat_imp = feat_imp.sort_values(ascending=False).head(15)

plt.figure(figsize=(10,6))
feat_imp.plot(kind='bar', color='green')
plt.title("Top 15 Important Features")
plt.tight_layout()
plt.show()